# SAM2-GHAN Final Dataset Builder
### Source: `segmented_raw/` from original SAM2 output zip

**কী হচ্ছে এই notebook-এ:**

| Step | কাজ | কারণ |
|------|-----|------|
| 1 | Config define | সব settings এক জায়গায় |
| 2 | `segmented_raw` extract | শুধু original SAM2 pairs, DROP_21 skip |
| 3 | Fresh 80/10/10 split | clean, unbiased split |
| 4 | Smart augmentation | `target = min(400, orig×8)` → no duplicates |
| 5 | Val/test fix | val<5 → val+test merge into test |
| 6 | নতুন CSV + JSON | class_id 0→203, genus_id 0→103, venom weights |
| 7 | Integrity check | 7টা verification |
| 8 | Zip final dataset | clean zip only |
| 9 | Delete temp files | disk free |

**Expected output:**
```
204 classes | 104 genera | ~85,300 total images
Train: ~79,440  Val: ~3,442  Test: ~3,431
```

## Step 1 — Config

In [1]:
import os, json, csv, shutil, gc, cv2, random, glob
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
SRC_ZIP  = "/kaggle/input/notebooks/zahidhasantonmoy/sam2-ghan-dataset-builder/_output_.zip"
META_CSV = "/kaggle/input/datasets/zahidhasantonmoy/snake-final-csv/final_fixed_metadata.csv"

WORK_DIR  = "/kaggle/working"
RAW_DIR   = "/kaggle/working/segmented_raw"   # extracted original pairs
FINAL_DIR = "/kaggle/working/SAM2_GHAN_Final" # final split+aug dataset
OUT_ZIP   = "/kaggle/working/SAM2_GHAN_Final_Dataset.zip"

# ── DROP LIST ─────────────────────────────────────────────────────────────────
# 7 critical: val/test only 1–2 images after split
# 14 extreme: <25 original images → aug ratio >16x → too many duplicates
DROP_21 = {
    'brown-spotted_whipsnake', 'rough-scaled_death_adder', 'pilbara_death_adder',
    'broad-banded_copperhead', 'saba_racer', 'blunt-headed_slug_snake',
    'madagascar_ground_boa',
    'chiapan_burrowing_snake', 'wynad_keelback', 'farnsworth_s_vine_snake',
    'arafura_file_snake', 'florida_cottonmouth', 'many-spotted_snake',
    'small-mouthed_blind_snake', 'western_bush_viper', 'malayan_green_whipsnake',
    'yucatecan_cantil', 'occipital_snake', 'olive_sea_snake', 'pronged_blind_snake',
    'reticulated_centipede-eater',
}

# ── AUGMENTATION ──────────────────────────────────────────────────────────────
AUG_MAX    = 400   # standard target
AUG_MULT   = 8     # unique transforms per original image
AUG_THRESH = 50    # apply smart rule if orig < this
WRITE_BATCH = 64

# ── SPLIT ─────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
VAL_MIN     = 5    # if val < VAL_MIN after split → merge val into test
SEED        = 42

random.seed(SEED); np.random.seed(SEED)

# Create directories
for split in ['train', 'val', 'test']:
    os.makedirs(f"{FINAL_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{FINAL_DIR}/{split}/masks",  exist_ok=True)
os.makedirs(RAW_DIR + "/images", exist_ok=True)
os.makedirs(RAW_DIR + "/masks",  exist_ok=True)

print("Config ready!")
print(f"  Source zip : {SRC_ZIP}")
print(f"  Drop list  : {len(DROP_21)} classes")
print(f"  Aug rule   : target = min({AUG_MAX}, orig × {AUG_MULT})  [if orig < {AUG_THRESH}]")
print(f"  Val fix    : merge val→test if val < {VAL_MIN}")

Config ready!
  Source zip : /kaggle/input/notebooks/zahidhasantonmoy/sam2-ghan-dataset-builder/_output_.zip
  Drop list  : 21 classes
  Aug rule   : target = min(400, orig × 8)  [if orig < 50]
  Val fix    : merge val→test if val < 5


## Step 2 — Extract `segmented_raw` Only

In [2]:
import zipfile

print("Extracting segmented_raw from source zip...")
print("Skipping: .virtual_documents, sam2/, SAM2_GHAN_Dataset_Ready/, DROP_21 classes\n")

extracted = skipped = 0

with zipfile.ZipFile(SRC_ZIP, 'r') as z:
    all_entries = z.namelist()
    print(f"Total entries in zip: {len(all_entries):,}")

    # Filter: only segmented_raw entries
    raw_entries = [e for e in all_entries
                   if 'segmented_raw' in e and not e.endswith('/')]
    print(f"segmented_raw entries: {len(raw_entries):,}")

    pbar = tqdm(raw_entries, desc="Extracting")
    for entry in pbar:
        parts = Path(entry).parts

        # Find class name in path
        # Structure: .../segmented_raw/images/class_name/file.jpg
        #        or: .../segmented_raw/masks/class_name/file_mask.png
        try:
            sr_idx = next(i for i, p in enumerate(parts) if p == 'segmented_raw')
            sub    = parts[sr_idx + 1]   # 'images' or 'masks'
            cls    = parts[sr_idx + 2]   # class name
        except (StopIteration, IndexError):
            skipped += 1
            continue

        # Skip dropped classes
        if cls in DROP_21:
            skipped += 1
            continue

        # Destination: RAW_DIR/images/class/file OR RAW_DIR/masks/class/file
        fname = parts[-1]
        dest  = Path(RAW_DIR) / sub / cls / fname
        dest.parent.mkdir(parents=True, exist_ok=True)

        with z.open(entry) as src, open(dest, 'wb') as dst:
            dst.write(src.read())
        extracted += 1

        if extracted % 5000 == 0:
            pbar.set_postfix(extracted=extracted, skipped=skipped)

print(f"\n=== EXTRACTION DONE ===")
print(f"Extracted : {extracted:,} files")
print(f"Skipped   : {skipped:,} files")

# Verify classes found
img_classes = sorted(
    d for d in os.listdir(RAW_DIR + '/images')
    if os.path.isdir(os.path.join(RAW_DIR + '/images', d))
)
print(f"Classes found in segmented_raw: {len(img_classes)}")
dropped_found = [c for c in img_classes if c in DROP_21]
print(f"Dropped classes present: {len(dropped_found)} (should be 0)")

Extracting segmented_raw from source zip...
Skipping: .virtual_documents, sam2/, SAM2_GHAN_Dataset_Ready/, DROP_21 classes

Total entries in zip: 266,209
segmented_raw entries: 69,716


Extracting:   0%|          | 0/69716 [00:00<?, ?it/s]


=== EXTRACTION DONE ===
Extracted : 68,754 files
Skipped   : 962 files
Classes found in segmented_raw: 204
Dropped classes present: 0 (should be 0)


## Step 3 — Fresh 80/10/10 Stratified Split

In [3]:
import os, random, shutil
from pathlib import Path
from tqdm.notebook import tqdm

random.seed(SEED)

RAW_IMG = RAW_DIR + '/images'
RAW_MSK = RAW_DIR + '/masks'

classes = sorted(
    d for d in os.listdir(RAW_IMG)
    if os.path.isdir(os.path.join(RAW_IMG, d)) and d not in DROP_21
)
print(f"Splitting {len(classes)} classes — ratio {TRAIN_RATIO}/{VAL_RATIO}/{1-TRAIN_RATIO-VAL_RATIO}\n")

split_records = {}   # class → {train: n, val: n, test: n}
small_val_set = set()
summary = defaultdict(int)

for cls in tqdm(classes, desc="Splitting"):
    img_dir = os.path.join(RAW_IMG, cls)
    msk_dir = os.path.join(RAW_MSK, cls)

    # Collect valid image-mask pairs
    all_imgs = sorted(f for f in os.listdir(img_dir)
                      if f.endswith(('.jpg', '.png')))
    valid = []
    for imgf in all_imgs:
        stem  = Path(imgf).stem
        maskf = stem + '_mask.png'
        if os.path.exists(os.path.join(msk_dir, maskf)):
            valid.append((imgf, maskf))

    n = len(valid)
    if n == 0:
        print(f"  WARN: {cls} has 0 valid pairs — skipping")
        continue

    random.shuffle(valid)

    # Calculate split sizes
    n_val   = max(1, round(n * VAL_RATIO))
    n_test  = max(1, round(n * (1 - TRAIN_RATIO - VAL_RATIO)))
    n_train = n - n_val - n_test
    if n_train < 1:
        n_train = max(1, n - 2)
        n_val   = 1
        n_test  = max(1, n - n_train - 1)

    # Track small val classes
    if n_val < VAL_MIN:
        small_val_set.add(cls)

    splits = {
        'train': valid[:n_train],
        'val'  : valid[n_train : n_train + n_val],
        'test' : valid[n_train + n_val :]
    }

    # Copy to FINAL_DIR split folders
    for sp, pairs in splits.items():
        di = os.path.join(FINAL_DIR, sp, 'images', cls)
        dm = os.path.join(FINAL_DIR, sp, 'masks',  cls)
        os.makedirs(di, exist_ok=True)
        os.makedirs(dm, exist_ok=True)
        for imgf, maskf in pairs:
            shutil.copy2(os.path.join(img_dir, imgf),  os.path.join(di, imgf))
            shutil.copy2(os.path.join(msk_dir, maskf), os.path.join(dm, maskf))
        summary[sp] += len(pairs)

    split_records[cls] = {'train': n_train, 'val': n_val, 'test': n_test, 'total': n}

total = sum(summary.values())
print(f"\n=== SPLIT DONE ===")
for sp in ['train', 'val', 'test']:
    print(f"  {sp:5s}: {summary[sp]:6,} images  ({summary[sp]/total*100:.1f}%)")
print(f"  Total: {total:,}")
print(f"\nSmall val classes (val<{VAL_MIN}): {len(small_val_set)}")
for c in sorted(small_val_set):
    r = split_records[c]
    print(f"  {c:<45}: val={r['val']}, test={r['test']} → will merge val→test")

Splitting 204 classes — ratio 0.8/0.1/0.09999999999999995



Splitting:   0%|          | 0/204 [00:00<?, ?it/s]


=== SPLIT DONE ===
  train: 27,503 images  (80.0%)
  val  :  3,442 images  (10.0%)
  test :  3,432 images  (10.0%)
  Total: 34,377

Small val classes (val<5): 17
  alpha_snake                                  : val=3, test=3 → will merge val→test
  blackish_blind_snake                         : val=3, test=3 → will merge val→test
  cape_centipede-eater                         : val=4, test=4 → will merge val→test
  cape_coral_snake                             : val=3, test=3 → will merge val→test
  childrens_python                             : val=3, test=3 → will merge val→test
  common_rough-sided_snake                     : val=3, test=3 → will merge val→test
  dumerils_boa                                 : val=3, test=3 → will merge val→test
  isabelline_whipsnake                         : val=4, test=4 → will merge val→test
  little_file_snake                            : val=4, test=4 → will merge val→test
  northern_death_adder                         : val=4, test=4 → will me

## Step 4 — Smart Augmentation (Train only)

In [4]:
import os, cv2, glob, random, gc
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

random.seed(SEED); np.random.seed(SEED)

TRAIN_IMG = os.path.join(FINAL_DIR, 'train', 'images')
TRAIN_MSK = os.path.join(FINAL_DIR, 'train', 'masks')


def augment_pair(img, mask):
    """Same spatial transform on image AND mask — 8 unique transforms."""
    c = random.randint(0, 7)
    if c == 0:
        return cv2.flip(img, 1), cv2.flip(mask, 1)
    elif c == 1:
        return cv2.flip(img, 0), cv2.flip(mask, 0)
    elif c == 2:
        return (cv2.rotate(img,  cv2.ROTATE_90_CLOCKWISE),
                cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE))
    elif c == 3:
        return (cv2.rotate(img,  cv2.ROTATE_90_COUNTERCLOCKWISE),
                cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE))
    elif c == 4:
        return (cv2.rotate(img,  cv2.ROTATE_180),
                cv2.rotate(mask, cv2.ROTATE_180))
    elif c == 5:
        f = random.uniform(0.75, 1.30)
        return np.clip(img.astype(np.float32)*f, 0, 255).astype(np.uint8), mask
    elif c == 6:
        f  = random.uniform(0.80, 1.20)
        fi = cv2.flip(img, 1)
        return np.clip(fi.astype(np.float32)*f, 0, 255).astype(np.uint8), cv2.flip(mask, 1)
    else:
        a = random.uniform(0.85, 1.15)
        b = random.randint(-15, 15)
        return np.clip(img.astype(np.float32)*a+b, 0, 255).astype(np.uint8), mask


classes = sorted(
    d for d in os.listdir(TRAIN_IMG)
    if os.path.isdir(os.path.join(TRAIN_IMG, d))
)
print(f"Augmenting {len(classes)} train classes...")
print(f"Rule: target = min({AUG_MAX}, orig × {AUG_MULT})  if orig < {AUG_THRESH}\n")

aug_log = {}
total_written = 0

for cls in tqdm(classes, desc="Augmenting"):
    img_dir  = os.path.join(TRAIN_IMG, cls)
    mask_dir = os.path.join(TRAIN_MSK, cls)

    img_files = sorted(glob.glob(os.path.join(img_dir, '*.jpg')) +
                       glob.glob(os.path.join(img_dir, '*.png')))

    # Load all valid original pairs
    pairs = []
    for imgf in img_files:
        stem  = Path(imgf).stem
        maskf = os.path.join(mask_dir, stem + '_mask.png')
        if not os.path.exists(maskf): continue
        img  = cv2.imread(imgf)
        msk  = cv2.imread(maskf, cv2.IMREAD_GRAYSCALE)
        if img is not None and msk is not None:
            pairs.append((img, msk, stem))

    orig_n = len(pairs)
    if orig_n == 0: continue

    # Smart target
    target     = min(AUG_MAX, orig_n * AUG_MULT) if orig_n < AUG_THRESH else AUG_MAX
    aug_needed = target - orig_n
    aug_log[cls] = {'orig': orig_n, 'target': target, 'aug': aug_needed}

    if aug_needed <= 0:
        del pairs; gc.collect()
        continue

    # Write augmented images in batches
    bi, bm = [], []
    for i in range(aug_needed):
        src_img, src_msk, src_stem = random.choice(pairs)
        a_img, a_msk = augment_pair(src_img, src_msk)
        name = f'aug_{i:05d}_{src_stem}'
        bi.append((a_img, os.path.join(img_dir,  name + '.jpg')))
        bm.append((a_msk, os.path.join(mask_dir, name + '_mask.png')))

        if len(bi) >= WRITE_BATCH:
            for im, fp in bi: cv2.imwrite(fp, im, [cv2.IMWRITE_JPEG_QUALITY, 95])
            for mk, fp in bm: cv2.imwrite(fp, mk)
            total_written += len(bi)
            bi.clear(); bm.clear(); gc.collect()

    for im, fp in bi: cv2.imwrite(fp, im, [cv2.IMWRITE_JPEG_QUALITY, 95])
    for mk, fp in bm: cv2.imwrite(fp, mk)
    total_written += len(bi)
    bi.clear(); bm.clear()
    del pairs; gc.collect()

print(f"\n=== AUGMENTATION DONE ===")
print(f"Total aug images written: {total_written:,}")
total_train = sum(os.path.isfile(os.path.join(TRAIN_IMG, cls, f))
                  for cls in os.listdir(TRAIN_IMG)
                  for f in os.listdir(os.path.join(TRAIN_IMG, cls))
                  if f.endswith(('.jpg','.png')))
print(f"Total train images now : {total_train:,}")

smart_classes = {c:v for c,v in aug_log.items() if v['target'] < AUG_MAX}
if smart_classes:
    print(f"\nSmart-aug classes ({len(smart_classes)}) — reduced target to avoid duplicates:")
    for c, v in sorted(smart_classes.items(), key=lambda x: x[1]['orig']):
        print(f"  {c:<45}: {v['orig']} orig → target={v['target']}")

Augmenting 204 train classes...
Rule: target = min(400, orig × 8)  if orig < 50



Augmenting:   0%|          | 0/204 [00:00<?, ?it/s]


=== AUGMENTATION DONE ===
Total aug images written: 50,937
Total train images now : 78,440

Smart-aug classes (19) — reduced target to avoid duplicates:
  alpha_snake                                  : 24 orig → target=192
  common_rough-sided_snake                     : 25 orig → target=200
  cape_coral_snake                             : 26 orig → target=208
  northern_vine_snake                          : 26 orig → target=208
  variable_bush_viper                          : 26 orig → target=208
  childrens_python                             : 27 orig → target=216
  shield-nosed_cobra                           : 27 orig → target=216
  spotted_python                               : 27 orig → target=216
  blackish_blind_snake                         : 28 orig → target=224
  dumerils_boa                                 : 28 orig → target=224
  northern_death_adder                         : 28 orig → target=224
  veliferum_snake                              : 28 orig → target=224
  zamb

## Step 5 — Val/Test Fix (merge small-val into test)

In [5]:
import os, shutil
from pathlib import Path

VAL_IMG  = os.path.join(FINAL_DIR, 'val',  'images')
VAL_MSK  = os.path.join(FINAL_DIR, 'val',  'masks')
TEST_IMG = os.path.join(FINAL_DIR, 'test', 'images')
TEST_MSK = os.path.join(FINAL_DIR, 'test', 'masks')

print(f"Fixing {len(small_val_set)} small-val classes (val<{VAL_MIN} → merge into test)\n")

merged_count = 0
for cls in sorted(small_val_set):
    vi = os.path.join(VAL_IMG, cls)
    vm = os.path.join(VAL_MSK, cls)
    ti = os.path.join(TEST_IMG, cls)
    tm = os.path.join(TEST_MSK, cls)

    if not os.path.isdir(vi): continue
    os.makedirs(ti, exist_ok=True)
    os.makedirs(tm, exist_ok=True)

    moved = 0
    for imgf in os.listdir(vi):
        if not imgf.endswith(('.jpg', '.png')): continue
        stem  = Path(imgf).stem
        maskf = stem + '_mask.png'
        shutil.move(os.path.join(vi, imgf),  os.path.join(ti, imgf))
        if os.path.exists(os.path.join(vm, maskf)):
            shutil.move(os.path.join(vm, maskf), os.path.join(tm, maskf))
        moved += 1

    # Remove now-empty val folder
    shutil.rmtree(vi, ignore_errors=True)
    shutil.rmtree(vm, ignore_errors=True)

    new_test = len([f for f in os.listdir(ti) if f.endswith(('.jpg','.png'))])
    print(f"  {cls:<45}: moved {moved} val → test (test now={new_test})")
    merged_count += moved

print(f"\nTotal files moved val→test: {merged_count}")

# Update split_records for small_val classes
for cls in small_val_set:
    if cls in split_records:
        split_records[cls]['test'] += split_records[cls]['val']
        split_records[cls]['val']   = 0

print("Val/test fix done!")

Fixing 17 small-val classes (val<5 → merge into test)

  alpha_snake                                  : moved 3 val → test (test now=6)
  blackish_blind_snake                         : moved 3 val → test (test now=6)
  cape_centipede-eater                         : moved 4 val → test (test now=8)
  cape_coral_snake                             : moved 3 val → test (test now=6)
  childrens_python                             : moved 3 val → test (test now=6)
  common_rough-sided_snake                     : moved 3 val → test (test now=6)
  dumerils_boa                                 : moved 3 val → test (test now=6)
  isabelline_whipsnake                         : moved 4 val → test (test now=8)
  little_file_snake                            : moved 4 val → test (test now=8)
  northern_death_adder                         : moved 4 val → test (test now=8)
  northern_vine_snake                          : moved 3 val → test (test now=6)
  red_pipe_snake                               : moved

## Step 6 — Build CSV + label_mappings.json

In [6]:
import csv, json, os
from collections import Counter

# Load original metadata
meta_raw = {}
with open(META_CSV) as f:
    for row in csv.DictReader(f):
        meta_raw[row['class_name']] = row

# Active classes from actual train directory (sorted alphabetically → new class_id)
active_classes = sorted(
    d for d in os.listdir(os.path.join(FINAL_DIR, 'train', 'images'))
    if os.path.isdir(os.path.join(FINAL_DIR, 'train', 'images', d))
)
print(f"Active classes : {len(active_classes)}")

# New class_id: 0 → 203 (alphabetical)
new_class_to_id = {cls: i for i, cls in enumerate(active_classes)}

# New genus_id: 0 → 103 (only active genera, alphabetical)
active_genera   = sorted(set(meta_raw[cls]['genus']
                             for cls in active_classes if cls in meta_raw))
new_genus_to_id = {g: i for i, g in enumerate(active_genera)}
print(f"Active genera  : {len(active_genera)}")

# Venom order
venom_order = {'Non-venomous':0,'Mildly venomous':1,'Mild':2,'Moderate':3,'Strong':4}

# Venom class weights (inverse frequency)
venom_counts = Counter(meta_raw[cls]['venom_category']
                       for cls in active_classes if cls in meta_raw)
n_total      = len(active_classes)
venom_weights = {
    vid: round(n_total / (len(venom_order) * venom_counts.get(cat, 1)), 4)
    for cat, vid in venom_order.items()
}

# Count actual files per class per split
def count_files(split, cls):
    d = os.path.join(FINAL_DIR, split, 'images', cls)
    if not os.path.isdir(d): return 0
    return len([f for f in os.listdir(d) if f.endswith(('.jpg','.png'))])

# Build CSV rows
new_rows = []
for cls in active_classes:
    if cls not in meta_raw: continue
    m        = meta_raw[cls]
    genus    = m['genus']
    new_rows.append({
        'class_id'               : new_class_to_id[cls],
        'class_name'             : cls,
        'scientific_name'        : m['scientific_name'],
        'genus'                  : genus,
        'genus_id'               : new_genus_to_id.get(genus, -1),
        'is_venomous'            : m['is_venomous'],
        'venom_category'         : m['venom_category'],
        'venom_id'               : venom_order.get(m['venom_category'], -1),
        'train_count'            : count_files('train', cls),
        'val_count'              : count_files('val',   cls),
        'test_count'             : count_files('test',  cls),
        'geographic_distribution': m['geographic_distribution'],
        'primary_habitat'        : m['primary_habitat'],
    })

# Save CSV
CSV_OUT = f"{FINAL_DIR}/metadata_hierarchy.csv"
fields  = ['class_id','class_name','scientific_name','genus','genus_id',
           'is_venomous','venom_category','venom_id',
           'train_count','val_count','test_count',
           'geographic_distribution','primary_habitat']
with open(CSV_OUT, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader(); w.writerows(new_rows)

# Save JSON
JSON_OUT = f"{FINAL_DIR}/label_mappings.json"
mappings = {
    'num_species'         : len(active_classes),
    'num_genera'          : len(active_genera),
    'num_venom_cats'      : len(venom_order),
    'class_to_id'         : new_class_to_id,
    'id_to_class'         : {str(v): k for k,v in new_class_to_id.items()},
    'genus_to_id'         : new_genus_to_id,
    'id_to_genus'         : {str(v): k for k,v in new_genus_to_id.items()},
    'venom_to_id'         : venom_order,
    'id_to_venom'         : {str(v): k for k,v in venom_order.items()},
    'class_to_genus_id'   : {cls: new_genus_to_id.get(meta_raw[cls]['genus'], -1)
                             for cls in active_classes if cls in meta_raw},
    'class_to_venom_id'   : {cls: venom_order.get(meta_raw[cls]['venom_category'], -1)
                             for cls in active_classes if cls in meta_raw},
    'venom_class_weights' : [venom_weights[i] for i in range(5)],
    'dropped_classes'     : sorted(DROP_21),
    'small_val_merged_to_test': sorted(small_val_set),
}
with open(JSON_OUT, 'w') as f:
    json.dump(mappings, f, indent=2)

# Summary
tot_tr = sum(r['train_count'] for r in new_rows)
tot_va = sum(r['val_count']   for r in new_rows)
tot_te = sum(r['test_count']  for r in new_rows)
print(f"\n{'='*45}")
print(f"  Species          : {len(active_classes)}")
print(f"  Genera           : {len(active_genera)}")
print(f"  Venom categories : {len(venom_order)}")
print(f"  Train total      : {tot_tr:,}")
print(f"  Val   total      : {tot_va:,}  ({sum(1 for r in new_rows if r['val_count']==0)} classes val=0)")
print(f"  Test  total      : {tot_te:,}")
print(f"  Grand total      : {tot_tr+tot_va+tot_te:,}")
print(f"{'='*45}")
print(f"  Venom weights    : {mappings['venom_class_weights']}")
print(f"  CSV  → {CSV_OUT}")
print(f"  JSON → {JSON_OUT}")

Active classes : 204
Active genera  : 104

  Species          : 204
  Genera           : 104
  Venom categories : 5
  Train total      : 78,440
  Val   total      : 3,385  (17 classes val=0)
  Test  total      : 3,489
  Grand total      : 85,314
  Venom weights    : [0.4636, 0.9951, 2.72, 1.8545, 1.0737]
  CSV  → /kaggle/working/SAM2_GHAN_Final/metadata_hierarchy.csv
  JSON → /kaggle/working/SAM2_GHAN_Final/label_mappings.json


## Step 7 — Full Integrity Verification

In [7]:
import os, json, csv
from pathlib import Path

print("=" * 60)
print("  INTEGRITY CHECK")
print("=" * 60)
all_ok = True

# [1] Image-mask pairs
print("\n[1] Image-mask pairs")
grand = 0
for sp in ['train','val','test']:
    ib = os.path.join(FINAL_DIR, sp, 'images')
    mb = os.path.join(FINAL_DIR, sp, 'masks')
    cls_list = sorted(d for d in os.listdir(ib)
                      if os.path.isdir(os.path.join(ib, d)))
    total = missing = 0
    counts = []
    for cls in cls_list:
        imgs = [f for f in os.listdir(os.path.join(ib, cls))
                if f.endswith(('.jpg','.png'))]
        for imgf in imgs:
            mf = Path(imgf).stem + '_mask.png'
            if not os.path.exists(os.path.join(mb, cls, mf)):
                missing += 1
        total  += len(imgs)
        counts.append(len(imgs))
    grand += total
    st = 'OK' if missing==0 else f'FAIL ({missing} missing)'
    if missing: all_ok = False
    mn = min(counts) if counts else 0
    mx = max(counts) if counts else 0
    print(f"   {sp:5s} | classes={len(cls_list):3d} | images={total:7,} | "
          f"min/max per class={mn}/{mx} | {st}")
print(f"   Total: {grand:,}")

# [2] No dropped class
print("\n[2] Dropped classes absent")
found = []
for sp in ['train','val','test']:
    for cls in os.listdir(os.path.join(FINAL_DIR, sp, 'images')):
        if cls in DROP_21: found.append(f"{sp}/{cls}")
if found: all_ok = False; print(f"   FAIL: {found}")
else: print(f"   OK — none of 21 dropped classes present")

# [3] class_id contiguous
print("\n[3] class_id 0→203 contiguous")
csv_rows = []
with open(f"{FINAL_DIR}/metadata_hierarchy.csv") as f:
    csv_rows = list(csv.DictReader(f))
ids = sorted(int(r['class_id']) for r in csv_rows)
if ids == list(range(len(csv_rows))):
    print(f"   OK — 0 to {max(ids)}, no gaps")
else:
    all_ok = False
    print(f"   FAIL — gaps found")

# [4] genus_id contiguous
print("\n[4] genus_id 0→103 contiguous")
gids = sorted(set(int(r['genus_id']) for r in csv_rows))
if gids == list(range(len(gids))): print(f"   OK — 0 to {max(gids)}")
else: all_ok = False; print(f"   FAIL")

# [5] CSV count matches actual files
print("\n[5] CSV counts match actual files")
mismatch = 0
for r in csv_rows:
    for sp, col in [('train','train_count'),('val','val_count'),('test','test_count')]:
        csv_n  = int(r[col])
        d      = os.path.join(FINAL_DIR, sp, 'images', r['class_name'])
        actual = len([f for f in os.listdir(d) if f.endswith(('.jpg','.png'))]) if os.path.isdir(d) else 0
        if csv_n != actual:
            mismatch += 1
            if mismatch <= 5:
                print(f"   MISMATCH {r['class_name']}/{sp}: csv={csv_n} actual={actual}")
if mismatch == 0: print(f"   OK — all counts match")
else: all_ok = False; print(f"   FAIL — {mismatch} mismatches")

# [6] JSON valid
print("\n[6] label_mappings.json")
with open(f"{FINAL_DIR}/label_mappings.json") as f:
    lm = json.load(f)
print(f"   num_species    : {lm['num_species']}")
print(f"   num_genera     : {lm['num_genera']}")
print(f"   venom_weights  : {lm['venom_class_weights']}")
print(f"   dropped_classes: {len(lm['dropped_classes'])}")

# [7] Train has no val/test images (no data leakage)
print("\n[7] No data leakage (train filenames not in val/test)")
# Quick sample check on 5 classes
leakage = 0
sample_classes = list(active_classes[:5])
for cls in sample_classes:
    tr_files = set(os.listdir(os.path.join(FINAL_DIR,'train','images',cls))
                   if os.path.isdir(os.path.join(FINAL_DIR,'train','images',cls)) else [])
    for sp in ['val','test']:
        sp_dir = os.path.join(FINAL_DIR, sp, 'images', cls)
        if not os.path.isdir(sp_dir): continue
        overlap = tr_files & set(os.listdir(sp_dir))
        # aug_ files are only in train, so overlap = original files in both
        real_overlap = [f for f in overlap if not f.startswith('aug_')]
        if real_overlap: leakage += len(real_overlap)
if leakage == 0: print(f"   OK — no leakage in sampled classes")
else: all_ok = False; print(f"   FAIL — {leakage} leaked files")

print("\n" + "="*60)
print("  ALL CHECKS PASSED! Ready to zip." if all_ok else "  SOME CHECKS FAILED — fix before zipping.")
print("="*60)

  INTEGRITY CHECK

[1] Image-mask pairs
   train | classes=204 | images= 78,440 | min/max per class=192/400 | OK
   val   | classes=187 | images=  3,385 | min/max per class=5/49 | OK
   test  | classes=204 | images=  3,489 | min/max per class=5/49 | OK
   Total: 85,314

[2] Dropped classes absent
   OK — none of 21 dropped classes present

[3] class_id 0→203 contiguous
   OK — 0 to 203, no gaps

[4] genus_id 0→103 contiguous
   OK — 0 to 103

[5] CSV counts match actual files
   OK — all counts match

[6] label_mappings.json
   num_species    : 204
   num_genera     : 104
   venom_weights  : [0.4636, 0.9951, 2.72, 1.8545, 1.0737]
   dropped_classes: 21

[7] No data leakage (train filenames not in val/test)
   OK — no leakage in sampled classes

  ALL CHECKS PASSED! Ready to zip.


## Step 8 — Zip Final Dataset

In [8]:
import os, subprocess

assert all_ok, "Integrity checks failed — fix before zipping!"

print("Zipping...  (5–15 min)")
ret = subprocess.run(['zip', '-r', '-q', OUT_ZIP, FINAL_DIR],
                     capture_output=True, text=True)

if ret.returncode != 0:
    print("Error:", ret.stderr)
else:
    gb = os.path.getsize(OUT_ZIP) / (1024**3)
    print(f"\nDone!")
    print(f"  File : SAM2_GHAN_Final_Dataset.zip")
    print(f"  Size : {gb:.2f} GB")
    print(f"\nContents:")
    print(f"  SAM2_GHAN_Final/")
    print(f"  ├── train/images+masks/  ~79,440 images (smart aug, no duplicates)")
    print(f"  ├── val/  images+masks/  ~3,385  images (original only)")
    print(f"  ├── test/ images+masks/  ~3,489  images (small-val merged)")
    print(f"  ├── metadata_hierarchy.csv  (204 classes, ID 0→203)")
    print(f"  └── label_mappings.json     (all dicts + venom weights)")

Zipping...  (5–15 min)

Done!
  File : SAM2_GHAN_Final_Dataset.zip
  Size : 3.16 GB

Contents:
  SAM2_GHAN_Final/
  ├── train/images+masks/  ~79,440 images (smart aug, no duplicates)
  ├── val/  images+masks/  ~3,385  images (original only)
  ├── test/ images+masks/  ~3,489  images (small-val merged)
  ├── metadata_hierarchy.csv  (204 classes, ID 0→203)
  └── label_mappings.json     (all dicts + venom weights)


## Step 9 — Delete Temp Files

In [9]:
import os, shutil
from pathlib import Path

zip_ok = os.path.exists(OUT_ZIP) and os.path.getsize(OUT_ZIP) > 1e8
assert zip_ok, "Zip not found or too small — NOT deleting!"

TO_DELETE = [
    RAW_DIR,      # extracted segmented_raw
    FINAL_DIR,    # final dir (now zipped)
    "/kaggle/working/sam2",
    "/kaggle/working/segmented_raw",
    "/kaggle/working/SAM2_GHAN_Dataset_Ready",
    "/kaggle/working/class_split_info.json",
    "/kaggle/working/split_records.json",
]

print("=" * 45)
freed = 0
for ps in TO_DELETE:
    p = Path(ps)
    if not p.exists(): print(f"  SKIP: {ps}"); continue
    size = (sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
            if p.is_dir() else p.stat().st_size)
    shutil.rmtree(p) if p.is_dir() else p.unlink()
    print(f"  DEL: {ps}  ({size/1e9:.2f} GB)")
    freed += size

print(f"\n  Freed: {freed/1e9:.2f} GB")
print("\n  /kaggle/working remaining:")
for item in sorted(os.listdir('/kaggle/working')):
    full = os.path.join('/kaggle/working', item)
    size = (os.path.getsize(full) if os.path.isfile(full)
            else sum(f.stat().st_size for f in Path(full).rglob('*') if f.is_file()))
    tag  = 'FILE' if os.path.isfile(full) else 'DIR'
    print(f"    [{tag}] {item}  ({size/1e9:.3f} GB)")

  DEL: /kaggle/working/segmented_raw  (1.47 GB)
  DEL: /kaggle/working/SAM2_GHAN_Final  (3.44 GB)
  SKIP: /kaggle/working/sam2
  SKIP: /kaggle/working/segmented_raw
  SKIP: /kaggle/working/SAM2_GHAN_Dataset_Ready
  SKIP: /kaggle/working/class_split_info.json
  SKIP: /kaggle/working/split_records.json

  Freed: 4.91 GB

  /kaggle/working remaining:
    [FILE] SAM2_GHAN_Final_Dataset.zip  (3.390 GB)
    [FILE] __notebook__.ipynb  (0.000 GB)
